# Advanced Problems with Solutions: Docstrings and Annotations

These problems focus on Python docstrings, annotations, introspection, runtime metadata, decorators, and best practices.

## Problem 1 — Build a Documentation Inspector

Write a function `describe_callable(fn)` that returns a dictionary containing:

- the callable name
- its cleaned docstring
- its signature as a string
- its annotations
- whether it has a return annotation

Use `inspect.signature` and `inspect.getdoc`.

In [1]:
import inspect

def describe_callable(fn):
    """Return useful documentation metadata for a callable."""
    sig = inspect.signature(fn)
    annotations = dict(getattr(fn, "__annotations__", {}))
    
    return {
        "name": getattr(fn, "__name__", type(fn).__name__),
        "docstring": inspect.getdoc(fn),
        "signature": str(sig),
        "annotations": annotations,
        "has_return_annotation": "return" in annotations
    }


def scale(value: float, factor: float = 1.0) -> float:
    """
    Scale a numeric value by a multiplier.
    
    Args:
        value: The value to scale.
        factor: The multiplier.
    
    Returns:
        The scaled value.
    """
    return value * factor


describe_callable(scale)

{'name': 'scale',
 'docstring': 'Scale a numeric value by a multiplier.\n\nArgs:\n    value: The value to scale.\n    factor: The multiplier.\n\nReturns:\n    The scaled value.',
 'signature': '(value: float, factor: float = 1.0) -> float',
 'annotations': {'value': float, 'factor': float, 'return': float},
 'has_return_annotation': True}

### Solution Notes

`inspect.getdoc()` is preferred over direct access to `.__doc__` because it cleans indentation and normalizes whitespace. `inspect.signature()` gives a reliable representation of parameters, defaults, `*args`, `**kwargs`, keyword-only arguments, and annotations.

## Problem 2 — Detect Undocumented Parameters

Write a function `missing_doc_params(fn)` that returns all function parameters that appear in the signature but do not appear in the function's docstring.

For this problem, use a simple rule: a parameter is documented if its name appears anywhere in the cleaned docstring.

In [2]:
import inspect

def missing_doc_params(fn):
    """Return parameter names that are not mentioned in the docstring."""
    doc = inspect.getdoc(fn) or ""
    sig = inspect.signature(fn)
    
    missing = []
    for name in sig.parameters:
        if name not in doc:
            missing.append(name)
    
    return missing


def connect(host: str, port: int, timeout: float = 3.0) -> bool:
    """
    Connect to a server.
    
    Args:
        host: Server hostname.
        port: Server port.
    """
    return True


missing_doc_params(connect)

['timeout']

### Solution Notes

This is intentionally simple. Real documentation linters should parse a specific docstring format such as Google style, NumPy style, or reStructuredText instead of checking raw substring membership.

## Problem 3 — Validate Runtime Types from Annotations

Write a decorator `enforce_simple_types` that checks positional and keyword arguments against simple annotations such as `int`, `str`, `float`, and `bool`.

The decorator should:

- preserve the original function metadata
- ignore parameters without annotations
- raise `TypeError` when an argument has the wrong type
- check the return value if a return annotation exists

In [3]:
import functools
import inspect

def enforce_simple_types(fn):
    """Enforce basic runtime type checks from simple annotations."""
    sig = inspect.signature(fn)
    annotations = fn.__annotations__
    
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        bound = sig.bind(*args, **kwargs)
        bound.apply_defaults()
        
        for name, value in bound.arguments.items():
            expected = annotations.get(name)
            if isinstance(expected, type) and not isinstance(value, expected):
                raise TypeError(
                    f"{name!r} must be {expected.__name__}, "
                    f"got {type(value).__name__}"
                )
        
        result = fn(*args, **kwargs)
        expected_return = annotations.get("return")
        
        if isinstance(expected_return, type) and not isinstance(result, expected_return):
            raise TypeError(
                f"return value must be {expected_return.__name__}, "
                f"got {type(result).__name__}"
            )
        
        return result
    
    return wrapper


@enforce_simple_types
def repeat(text: str, times: int = 2) -> str:
    """Repeat text a fixed number of times."""
    return text * times


print(repeat("ha", 3))

try:
    repeat("ha", "3")
except TypeError as exc:
    print(exc)

print(repeat.__name__)
print(repeat.__doc__)

hahaha
'times' must be int, got str
repeat
Repeat text a fixed number of times.


### Solution Notes

`functools.wraps` is essential because decorators otherwise hide the original function's name, docstring, annotations, and other metadata. This decorator intentionally handles only simple runtime-checkable annotations.

## Problem 4 — Preserve Annotations Through a Decorator

A decorator often replaces a function with a wrapper. Demonstrate what goes wrong without `functools.wraps`, then fix it.

Your solution should show the difference in `__name__`, `__doc__`, and `__annotations__`.

In [4]:
import functools

def bad_logger(fn):
    def wrapper(*args, **kwargs):
        print(f"Calling {fn.__name__}")
        return fn(*args, **kwargs)
    return wrapper


def good_logger(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        print(f"Calling {fn.__name__}")
        return fn(*args, **kwargs)
    return wrapper


@bad_logger
def add_bad(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


@good_logger
def add_good(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


print("BAD:")
print(add_bad.__name__)
print(add_bad.__doc__)
print(add_bad.__annotations__)

print("\nGOOD:")
print(add_good.__name__)
print(add_good.__doc__)
print(add_good.__annotations__)

BAD:
wrapper
None
{}

GOOD:
add_good
Add two integers.
{'a': <class 'int'>, 'b': <class 'int'>, 'return': <class 'int'>}


### Solution Notes

Without `functools.wraps`, introspection tools see the wrapper, not the original function. With `functools.wraps`, metadata such as `__name__`, `__doc__`, and `__annotations__` is copied to the wrapper.

## Problem 5 — Use `typing.get_type_hints`

Annotations may be strings, especially when forward references are used. Write a function `resolved_annotations(fn)` that returns evaluated annotations using `typing.get_type_hints`.

Then compare it with `fn.__annotations__`.

In [5]:
from typing import get_type_hints

class User:
    """A simple user model."""
    def __init__(self, name: str):
        self.name = name


def make_user(name: str) -> "User":
    """Create a User instance from a name."""
    return User(name)


def resolved_annotations(fn):
    """Return annotations with forward references resolved."""
    return get_type_hints(fn)


print("Raw annotations:")
print(make_user.__annotations__)

print("\nResolved annotations:")
print(resolved_annotations(make_user))

Raw annotations:
{'name': <class 'str'>, 'return': 'User'}

Resolved annotations:
{'name': <class 'str'>, 'return': <class '__main__.User'>}


### Solution Notes

`__annotations__` stores the raw annotation objects. `typing.get_type_hints()` resolves forward references and is usually the better choice for tools that need to interpret annotations semantically.

## Problem 6 — Generate Markdown Documentation Automatically

Write a function `markdown_docs(fn)` that returns Markdown documentation for a function.

Include:

- function name
- signature
- docstring
- parameter annotations
- return annotation

In [6]:
import inspect

def format_annotation(value):
    """Return a readable string for an annotation."""
    if hasattr(value, "__name__"):
        return value.__name__
    return repr(value)


def markdown_docs(fn):
    """Generate Markdown documentation for a function."""
    sig = inspect.signature(fn)
    doc = inspect.getdoc(fn) or "No docstring provided."
    annotations = getattr(fn, "__annotations__", {})
    
    lines = [
        f"# `{fn.__name__}`",
        "",
        "```python",
        f"{fn.__name__}{sig}",
        "```",
        "",
        doc,
        "",
        "## Annotations",
        ""
    ]
    
    for name, value in annotations.items():
        if name != "return":
            lines.append(f"- `{name}`: `{format_annotation(value)}`")
    
    if "return" in annotations:
        lines.append(f"- `return`: `{format_annotation(annotations['return'])}`")
    else:
        lines.append("- `return`: not annotated")
    
    return "\n".join(lines)


def normalize(text: str, lowercase: bool = True) -> str:
    """Strip surrounding whitespace and optionally lowercase the text."""
    text = text.strip()
    return text.lower() if lowercase else text


print(markdown_docs(normalize))

# `normalize`

```python
normalize(text: str, lowercase: bool = True) -> str
```

Strip surrounding whitespace and optionally lowercase the text.

## Annotations

- `text`: `str`
- `lowercase`: `bool`
- `return`: `str`


### Solution Notes

This pattern is the foundation of documentation generators. Production tools need stronger parsing, support for classes and modules, and careful formatting for complex annotations such as `list[str]`, `dict[str, int]`, and `Callable`.

## Problem 7 — Compare Docstring Claims with Annotations

Write a function `audit_docstring_return(fn)` that checks whether the word `Returns` appears in the docstring when the function has a return annotation.

Return one of:

- `'ok'`
- `'missing Returns section'`
- `'no return annotation'`

In [7]:
import inspect

def audit_docstring_return(fn):
    """Audit whether a return annotation is documented."""
    annotations = getattr(fn, "__annotations__", {})
    doc = inspect.getdoc(fn) or ""
    
    if "return" not in annotations:
        return "no return annotation"
    
    if "Returns" not in doc and "Return" not in doc:
        return "missing Returns section"
    
    return "ok"


def area(radius: float) -> float:
    """
    Calculate the area of a circle.
    
    Args:
        radius: Circle radius.
    """
    return 3.14159 * radius ** 2


def perimeter(width: float, height: float) -> float:
    """
    Calculate the perimeter of a rectangle.
    
    Returns:
        The perimeter.
    """
    return 2 * (width + height)


print(audit_docstring_return(area))
print(audit_docstring_return(perimeter))

missing Returns section
ok


### Solution Notes

This is a lightweight documentation quality check. A stricter version would parse the docstring format and verify that the documented return type matches the annotation.

## Problem 8 — Annotate `*args` and `**kwargs`

Create a function `summarize_scores` that accepts:

- a required `name: str`
- any number of positional scores annotated as `float`
- keyword metadata annotated as `str`

Return a dictionary containing the name, average score, and metadata.

Then inspect the function's signature and annotations.

In [8]:
import inspect

def summarize_scores(name: str, *scores: float, **metadata: str) -> dict:
    """
    Summarize a student's scores.
    
    Args:
        name: Student name.
        *scores: Numeric scores.
        **metadata: Extra string metadata.
    
    Returns:
        A dictionary with name, average, and metadata.
    """
    average = sum(scores) / len(scores) if scores else None
    return {
        "name": name,
        "average": average,
        "metadata": metadata
    }


print(inspect.signature(summarize_scores))
print(summarize_scores.__annotations__)
print(summarize_scores("Ada", 91.0, 95.5, course="Python", level="advanced"))

(name: str, *scores: float, **metadata: str) -> dict
{'name': <class 'str'>, 'scores': <class 'float'>, 'metadata': <class 'str'>, 'return': <class 'dict'>}
{'name': 'Ada', 'average': 93.25, 'metadata': {'course': 'Python', 'level': 'advanced'}}


### Solution Notes

Annotations on `*args` describe each collected positional argument, not the tuple object itself. Annotations on `**kwargs` describe each collected keyword value, not the dictionary object itself.

## Problem 9 — Create a Docstring Quality Score

Write a function `doc_quality_score(fn)` that gives a function up to 5 points:

1. has a docstring
2. docstring has at least 20 characters
3. all parameters have annotations
4. has a return annotation
5. all parameters are mentioned in the docstring

Return both the score and a list of failed checks.

In [9]:
import inspect

def doc_quality_score(fn):
    """Return a documentation quality score and failed checks."""
    score = 0
    failed = []
    
    doc = inspect.getdoc(fn) or ""
    sig = inspect.signature(fn)
    annotations = getattr(fn, "__annotations__", {})
    param_names = list(sig.parameters)
    
    checks = [
        (bool(doc), "missing docstring"),
        (len(doc) >= 20, "docstring too short"),
        (all(name in annotations for name in param_names), "not all parameters annotated"),
        ("return" in annotations, "missing return annotation"),
        (all(name in doc for name in param_names), "not all parameters mentioned in docstring")
    ]
    
    for passed, message in checks:
        if passed:
            score += 1
        else:
            failed.append(message)
    
    return score, failed


def divide(a: float, b: float) -> float:
    """
    Divide a by b.
    
    Args:
        a: Numerator.
        b: Denominator.
    
    Returns:
        The quotient.
    """
    return a / b


def weak(x, y):
    """Add."""
    return x + y


print(doc_quality_score(divide))
print(doc_quality_score(weak))

(5, [])
(1, ['docstring too short', 'not all parameters annotated', 'missing return annotation', 'not all parameters mentioned in docstring'])


### Solution Notes

A scoring function like this is useful for teaching and automated review. In real projects, tools such as linters, type checkers, and documentation generators provide deeper validation.

## Problem 10 — Advanced: A Mini Documentation Registry

Create a decorator `register_documented` that registers only functions that satisfy these rules:

- has a docstring
- has annotations for all parameters
- has a return annotation

Store accepted functions in a dictionary called `registry` using the function name as the key. If validation fails, raise `ValueError` with a helpful message.

In [10]:
import inspect

registry = {}

def register_documented(fn):
    """Register a function only if it is documented and annotated."""
    doc = inspect.getdoc(fn)
    sig = inspect.signature(fn)
    annotations = getattr(fn, "__annotations__", {})
    
    missing = []
    
    if not doc:
        missing.append("docstring")
    
    for name in sig.parameters:
        if name not in annotations:
            missing.append(f"annotation for parameter {name!r}")
    
    if "return" not in annotations:
        missing.append("return annotation")
    
    if missing:
        details = ", ".join(missing)
        raise ValueError(f"Cannot register {fn.__name__}: missing {details}")
    
    registry[fn.__name__] = fn
    return fn


@register_documented
def slugify(title: str) -> str:
    """Convert a title into a lowercase URL-friendly slug."""
    return title.strip().lower().replace(" ", "-")


print(registry)
print(registry["slugify"]("Hello Python World"))

try:
    @register_documented
    def broken(x):
        return x
except ValueError as exc:
    print(exc)

{'slugify': <function slugify at 0x000001AC9AA93A60>}
hello-python-world
Cannot register broken: missing docstring, annotation for parameter 'x', return annotation


### Solution Notes

This pattern combines decorators, annotations, docstrings, and introspection. It is useful for plugin systems, command registries, API frameworks, and documentation-driven development.